In [1]:
from PyDI.io import load_parquet
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import StringComparator, NumericComparator
from PyDI.entitymatching import RuleBasedMatcher
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust

# --------------------------------
# Prepare Data
# --------------------------------

# Define dataset paths
DATA_DIR = "../parquet/"

# Load Kaggle 380k dataset
kaggle380k = load_parquet(
    DATA_DIR + "kaggle380k.parquet",
    name="kaggle380k",
)

# Load Uber Eats dataset  
uber_eats = load_parquet(
    DATA_DIR + "uber_eats.parquet",
    name="uber_eats",
)

# Load Yelp dataset
yelp = load_parquet(
    DATA_DIR + "yelp.parquet",
    name="yelp",
)

# create id columns
kaggle380k['id'] = kaggle380k['kaggle380k_id']
uber_eats['id'] = uber_eats['uber_eats_id']
yelp['id'] = yelp['yelp_id']


# --------------------------------
# Perform Entity Matching
# --------------------------------

# define blocking
blocker_k2u = StandardBlocker(
    kaggle380k, uber_eats,
    on=['city'],
    batch_size=1000,
    output_dir="output/blocking-evaluation",
    id_column='id'
)

blocker_k2y = StandardBlocker(
    kaggle380k, yelp,
    on=['city'],
    batch_size=1000,
    output_dir="output/blocking-evaluation",
    id_column='id'
)

# define comparators
comparators = [
    # Name similarity
    StringComparator(
        column='name_norm',
        similarity_function='jaccard', 
        # no preprocessing needed
    ),
    
    # street name similarity
    StringComparator(
        column='street',
        similarity_function='jaccard', 
        preprocess=str.lower,
    ),

    # house number similarity
    NumericComparator(
        column='house_number',
        max_difference=2,
    ),

    # category similarity - supporting evidence
    StringComparator(
        column='categories',
        similarity_function='jaccard',
        preprocess=str.lower,
        list_strategy='concatenate' # Handle list attribute by concatenation
    )
]

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

rb_correspondences_k2u = matcher.match(
    df_left=kaggle380k,
    df_right=uber_eats, 
    candidates=blocker_k2u,
    comparators=comparators,
    weights=[0.5, 0.2, 0.2, 0.1], 
    threshold=0.7,
    id_column='id'
)

rb_correspondences_k2y = matcher.match(
    df_left=kaggle380k,
    df_right=yelp, 
    candidates=blocker_k2y,
    comparators=comparators,
    weights=[0.5, 0.2, 0.2, 0.1],  
    threshold=0.7,
    id_column='id'
)

# --------------------------------
# Data Fusion
# --------------------------------
# set trust scores
kaggle380k.attrs["trust_score"] = 3
uber_eats.attrs["trust_score"] = 2
yelp.attrs["trust_score"] = 1

# merge rule based correspondences
all_rb_correspondences = pd.concat([rb_correspondences_k2u, rb_correspondences_k2y], ignore_index=True)

# define data fusion strategy
strategy = DataFusionStrategy('usa_restaurant_fusion_strategy')

strategy.add_attribute_fuser('name', longest_string)
strategy.add_attribute_fuser('street', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('house_number', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('city', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('state', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('postal_code', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('country', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('latitude', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('longitude', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('categories', union)

# run fusion
engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

# fuse rule based matches
rb_fused = engine.run(
    datasets=[kaggle380k, uber_eats, yelp],
    correspondences=all_rb_correspondences,
    id_column="id",
    include_singletons=False,
)